In [41]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import pygeohash as pgh
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from math import radians, cos, sin, asin, sqrt
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

In [42]:
BASE_DIR = Path("dataset")

TRAIN_PATH = BASE_DIR / "train.csv"
TEST_PATH  = BASE_DIR / "test.csv"
TRUE_VALUES_PATH = BASE_DIR / "y_true.csv"

In [43]:
# ─────────────────────────────────────────────────────────────
# 1. SPATIAL MATH & UTILITIES
# ─────────────────────────────────────────────────────────────

def build_neighbor_map(all_geohashes, loc_demand_map):
    """Uses pygeohash to shift lat/lon and find 4 immediate neighbors."""
    DELTA = 0.0055
    gh_set = set(all_geohashes)
    neighbor_means = {}
    
    for gh in all_geohashes:
        lat, lon = pgh.decode(gh)
        neighbors = []
        # Shift North, South, East, West
        for dlat, dlon in [(DELTA,0), (-DELTA,0), (0,DELTA), (0,-DELTA)]:
            candidate = pgh.encode(lat+dlat, lon+dlon, precision=6)
            if candidate in gh_set and candidate != gh:
                neighbors.append(candidate)
                
        if neighbors:
            vals = [loc_demand_map.get(n, np.nan) for n in neighbors]
            vals = [v for v in vals if not np.isnan(v)]
            neighbor_means[gh] = np.mean(vals) if vals else np.nan
        else:
            neighbor_means[gh] = np.nan
            
    return neighbor_means

def haversine(lon1, lat1, lon2, lat2):
    """Calculate the great circle distance in kilometers."""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 
    return c * r

In [44]:
# ─────────────────────────────────────────────────────────────
# 2. LOAD & MERGE
# ─────────────────────────────────────────────────────────────
print("Loading data...")
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

df_train['is_train'] = 1
df_test['is_train']  = 0
df_test['demand']    = np.nan

df_full = pd.concat([df_train, df_test], ignore_index=True)

Loading data...


In [45]:
# ─────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
print("Engineering features...")

df_full['RoadType'] = df_full['RoadType'].fillna('Unknown')
df_train['RoadType'] = df_train['RoadType'].fillna('Unknown')

# A. Cyclical Time
df_full['hour']   = df_full['timestamp'].str.split(':').str[0].astype(int)
df_full['minute'] = df_full['timestamp'].str.split(':').str[1].astype(int)
mins_of_day       = df_full['hour'] * 60 + df_full['minute']

df_full['time_sin'] = np.sin(2 * np.pi * mins_of_day / 1440)
df_full['time_cos'] = np.cos(2 * np.pi * mins_of_day / 1440)
df_full['time_bucket'] = mins_of_day // 15

df_full['is_rush_hour'] = (
    ((df_full['hour'] >= 7) & (df_full['hour'] < 9)) |
    ((df_full['hour'] >= 17) & (df_full['hour'] < 19))
).astype(int)

df_full['is_peak_window'] = (
    (df_full['hour'] >= 6) & (df_full['hour'] < 20)
).astype(int)

# B. Geo Features Extraction (Using PyGeohash)
print("Extracting coordinates via PyGeohash...")
df_full['lat'] = df_full['geohash'].apply(lambda x: pgh.decode(x)[0])
df_full['lon'] = df_full['geohash'].apply(lambda x: pgh.decode(x)[1])

le = LabelEncoder()
df_full['geohash_cat'] = le.fit_transform(df_full['geohash'])
df_full['geohash_cat'] = df_full['geohash_cat'].astype('category')

# Distance to Center
busiest_geohash = df_train.groupby('geohash')['demand'].mean().idxmax()
center_lat, center_lon = pgh.decode(busiest_geohash)
df_full['dist_to_center'] = df_full.apply(
    lambda row: haversine(row['lon'], row['lat'], center_lon, center_lat), axis=1
)

# C. NEW: Spatial Clustering (Making something out of lat/lon)
print("Creating Spatial Zones using K-Means...")
kmeans = KMeans(n_clusters=20, random_state=42)
df_full['zone_id'] = kmeans.fit_predict(df_full[['lat', 'lon']])
df_full['zone_id'] = df_full['zone_id'].astype('category')

Engineering features...
Extracting coordinates via PyGeohash...
Creating Spatial Zones using K-Means...


In [46]:
# D. Safe Lag (demand_yesterday)
lookup_dict = df_train.set_index(['geohash', 'timestamp', 'day'])['demand'].to_dict()
df_full['demand_yesterday'] = df_full.apply(
    lambda row: lookup_dict.get((row['geohash'], row['timestamp'], row['day'] - 1), np.nan),
    axis=1
)

# E. Aggregations (Train only)
loc_stats = df_train.groupby('geohash')['demand'].agg(
    loc_mean='mean', loc_std='std', loc_median='median',
    loc_max='max', loc_p25=lambda x: x.quantile(0.25),
    loc_p75=lambda x: x.quantile(0.75)
).reset_index()

ts_stats = df_train.groupby('timestamp')['demand'].agg(
    ts_mean='mean', ts_std='std', ts_median='median', ts_max='max'
).reset_index()

# F. Road-Time Interaction Curve
df_train['time_bucket'] = (
    df_train['timestamp'].str.split(':').str[0].astype(int) * 60 + 
    df_train['timestamp'].str.split(':').str[1].astype(int)
) // 15
road_ts_stats = df_train.groupby(['RoadType', 'time_bucket'])['demand'].agg(
    road_ts_mean='mean'
).reset_index()

df_full = df_full.merge(loc_stats, on='geohash', how='left')
df_full = df_full.merge(ts_stats, on='timestamp', how='left')
df_full = df_full.merge(road_ts_stats, on=['RoadType', 'time_bucket'], how='left')

stat_cols = ['loc_mean', 'loc_std', 'loc_median', 'loc_max', 'loc_p25', 'loc_p75',
             'ts_mean', 'ts_std', 'ts_median', 'ts_max', 'road_ts_mean']
df_full[stat_cols] = df_full[stat_cols].fillna(0)
df_full['demand_yesterday'] = df_full['demand_yesterday'].fillna(df_full['loc_mean'])
# df_full['road_ts_mean'] = df_full['road_ts_mean'].replace(0, np.nan).fillna(df_full['ts_mean'])

# G. Spatial Spillover (Neighbors)
print("Computing geohash neighbor features...")
all_geohashes = df_full['geohash'].unique()
loc_demand_map = df_train.groupby('geohash')['demand'].mean().to_dict()
neighbor_map = build_neighbor_map(all_geohashes, loc_demand_map)
df_full['neighbor_mean_demand'] = df_full['geohash'].map(neighbor_map)
df_full['neighbor_mean_demand'] = df_full['neighbor_mean_demand'].fillna(df_full['loc_mean'])

# H. Ratios
eps = 1e-6
df_full['momentum_ratio']  = df_full['demand_yesterday'] / (df_full['loc_mean'] + eps)
df_full['ts_vs_loc']       = df_full['ts_mean']          / (df_full['loc_mean'] + eps)
df_full['neighbor_vs_loc'] = df_full['neighbor_mean_demand'] / (df_full['loc_mean'] + eps)
# df_full['road_vs_loc']     = df_full['road_ts_mean']     / (df_full['loc_mean'] + eps)
df_full['iqr_range']       = df_full['loc_p75'] - df_full['loc_p25']

# I. Categoricals & Bins
df_full['RoadType']      = df_full['RoadType'].astype('category')
df_full['Weather']       = df_full['Weather'].fillna('Unknown').astype('category')
df_full['LargeVehicles'] = df_full['LargeVehicles'].fillna('Unknown').astype('category')
df_full['Landmarks']     = df_full['Landmarks'].fillna('Unknown').astype('category')
df_full['NumberofLanes'] = df_full['NumberofLanes'].fillna(df_full['NumberofLanes'].median())
df_full['Temperature']   = df_full['Temperature'].fillna(df_full['Temperature'].median())
df_full['temp_bin']      = pd.cut(df_full['Temperature'], bins=[-np.inf, 5, 15, 25, 35, np.inf], labels=[0, 1, 2, 3, 4]).astype(int)

Computing geohash neighbor features...


In [47]:
# ─────────────────────────────────────────────────────────────
# 4. SPLIT & LOG TRANSFORM
# ─────────────────────────────────────────────────────────────
train_df = df_full[(df_full['is_train'] == 1) & (df_full['demand'].notnull())].copy()
test_df  = df_full[df_full['is_train'] == 0].copy()

FEATURES = [
    'time_sin', 'time_cos', 'time_bucket', 'is_rush_hour', 'is_peak_window',
    'lat', 'lon', 'dist_to_center', 'geohash_cat', 'zone_id', # <-- ALL SPATIAL FEATURES
    'loc_mean', 'loc_std', 'loc_median', 'loc_max', 'loc_p25', 'loc_p75', 'iqr_range',
    'ts_mean', 'ts_std', 'ts_median', 'ts_max',
    # 'road_ts_mean',
    'neighbor_mean_demand',
    'demand_yesterday',
    'momentum_ratio', 'ts_vs_loc', 'neighbor_vs_loc', # 'road_vs_loc',
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
    'Weather', 'Temperature', 'temp_bin',
]

X = train_df[FEATURES]
y = np.log1p(train_df['demand'])  
X_test = test_df[FEATURES]

print(f"\nTraining on {len(X)} rows with {len(FEATURES)} features. Predicting {len(X_test)} rows.")


Training on 77299 rows with 33 features. Predicting 41778 rows.


In [48]:
# ─────────────────────────────────────────────────────────────
# 5. 5-FOLD CV WITH OOF TARGET ENCODING
# ─────────────────────────────────────────────────────────────
print("\nStarting 5-Fold LightGBM Training...")

params = {
    'objective':       'regression',
    'metric':          'rmse',
    'learning_rate':   0.015,
    'max_depth':       7,
    'num_leaves':      48,          
    'feature_fraction': 0.65,       
    'bagging_fraction': 0.80,
    'bagging_freq':    3,
    'min_data_in_leaf': 50,         
    'lambda_l1':       1.0,         
    'lambda_l2':       2.0,         
    'verbose':        -1,
    'seed':            42
}

kf = KFold(n_splits=5)
oof_preds  = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))
train_df = train_df.reset_index(drop=True)
X = train_df[FEATURES].copy()

SMOOTH = 10  

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr_fold = X.iloc[train_idx].copy()
    y_tr_fold = y.iloc[train_idx]
    X_va_fold = X.iloc[val_idx].copy()
    X_te_fold = X_test.copy()

    # OOF Geohash Target Encoding
    gh_col_train = train_df['geohash'].iloc[train_idx]
    enc_df = pd.DataFrame({'geohash': gh_col_train, 'target': y_tr_fold.values})
    global_mean = y_tr_fold.mean()

    gh_stats = enc_df.groupby('geohash')['target'].agg(['mean', 'count'])
    gh_stats['encoded'] = (
        (gh_stats['mean'] * gh_stats['count'] + global_mean * SMOOTH)
        / (gh_stats['count'] + SMOOTH)
    )
    gh_enc_map = gh_stats['encoded'].to_dict()

    gh_val  = train_df['geohash'].iloc[val_idx]
    gh_test = test_df['geohash']

    X_tr_fold['geohash_target_enc'] = gh_col_train.map(gh_enc_map).fillna(global_mean).values
    X_va_fold['geohash_target_enc'] = gh_val.map(gh_enc_map).fillna(global_mean).values
    X_te_fold['geohash_target_enc'] = gh_test.map(gh_enc_map).fillna(global_mean).values

    feat_cols = FEATURES + ['geohash_target_enc']

    train_ds = lgb.Dataset(X_tr_fold[feat_cols], label=y_tr_fold)
    val_ds   = lgb.Dataset(X_va_fold[feat_cols], label=y.iloc[val_idx], reference=train_ds)

    model = lgb.train(
        params,
        train_ds,
        num_boost_round=6000, 
        valid_sets=[train_ds, val_ds],
        callbacks=[
            lgb.early_stopping(stopping_rounds=150, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )

    fold_preds = np.expm1(model.predict(X_va_fold[feat_cols]))
    oof_preds[val_idx] = fold_preds
    test_preds += np.expm1(model.predict(X_te_fold[feat_cols])) / kf.n_splits
    
    fold_r2 = r2_score(np.expm1(y.iloc[val_idx]), fold_preds)
    print(f"Fold {fold+1} R2: {fold_r2:.4f} (Trees: {model.best_iteration})")

overall_r2 = r2_score(np.expm1(y), oof_preds)
print(f"\n=> 5-Fold CV Overall R2: {overall_r2:.4f}")

# Feature importance 
feat_cols_final = FEATURES + ['geohash_target_enc']

fi = pd.DataFrame({
    'Feature': feat_cols_final,
    'Gain':    model.feature_importance(importance_type='gain')
}).sort_values('Gain', ascending=False)

print("\nTop 15 Features by Gain:")
print(fi.head(15).to_string(index=False))


Starting 5-Fold LightGBM Training...
Fold 1 R2: 0.8356 (Trees: 146)
Fold 2 R2: 0.9553 (Trees: 3603)
Fold 3 R2: 0.9584 (Trees: 4200)
Fold 4 R2: 0.9016 (Trees: 4143)
Fold 5 R2: 0.7508 (Trees: 229)

=> 5-Fold CV Overall R2: 0.8967

Top 15 Features by Gain:
           Feature         Gain
          RoadType 10575.001440
geohash_target_enc  2566.164041
          loc_mean  1864.327253
           loc_p75  1263.819569
         ts_median   403.152668
     LargeVehicles   371.070083
  demand_yesterday   339.171583
           ts_mean   328.978684
       geohash_cat   279.866748
            ts_std   238.546952
    momentum_ratio   187.830387
        loc_median   185.577242
       time_bucket    95.966555
           loc_p25    79.624817
           loc_max    51.276496


In [49]:
# ─────────────────────────────────────────────────────────────
# 6. STRICT SUBMISSION
# ─────────────────────────────────────────────────────────────
print("\nCreating submission...")
test_preds = np.maximum(test_preds, 0)

submission = pd.read_csv(TEST_PATH)[['Index']].copy()
pred_map   = dict(zip(test_df['Index'].values, test_preds))
submission['demand'] = submission['Index'].map(pred_map)

missing = submission['demand'].isna().sum()
if missing > 0:
    print(f"Warning: Filling {missing} missing rows with global train mean.")
    submission['demand'] = submission['demand'].fillna(np.expm1(y).mean())


Creating submission...


In [50]:
# ─────────────────────────────────────────────────────────────
# 7. Compare with True Values
# ─────────────────────────────────────────────────────────────
true_values_df = pd.read_csv(TRUE_VALUES_PATH)

comparison_df = submission.merge(true_values_df, on='Index', suffixes=('_pred', '_true'))

In [51]:
y_true = comparison_df['demand_true']
y_pred = comparison_df['demand_pred']

r2 = r2_score(y_true, y_pred)

print("\nEvaluation Metrics")
print(f"R²   : {r2:.6f}")


Evaluation Metrics
R²   : 0.900627


In [52]:
submission.to_csv("submission_final.csv", index=False)
print(f"Saved submission_final.csv ({len(submission)} rows)")

Saved submission_final.csv (41778 rows)


In [53]:
# import plotly.express as px
# import pygeohash as pgh

# # 1. Calculate the average demand for every unique geohash in your training data
# geo_map_df = df_train.groupby('geohash')['demand'].mean().reset_index()

# # 2. Decode the coordinates using the pygeohash library
# geo_map_df['lat'] = geo_map_df['geohash'].apply(lambda x: pgh.decode(x)[0])
# geo_map_df['lon'] = geo_map_df['geohash'].apply(lambda x: pgh.decode(x)[1])

# # 3. Render the interactive map
# fig = px.scatter_mapbox(
#     geo_map_df, 
#     lat="lat", 
#     lon="lon", 
#     color="demand",                         # Heatmap coloring based on busyness
#     color_continuous_scale="Plasma", 
#     size="demand",                          # Bubbles get larger with higher demand
#     zoom=10, 
#     mapbox_style="carto-positron",          # Uses a clean, street-level base map
#     title="Average Traffic Demand by Geohash"
# )

# # 4. Increase the size of the plot for better visibility
# fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0}, height=700)
# fig.show(renderer="notebook")

In [54]:
print(train_df[['day', 'timestamp']].head(20))
print(train_df[['day', 'timestamp']].tail(20))

    day timestamp
0    48       0:0
1    48       0:0
2    48       0:0
3    48       0:0
4    48       0:0
5    48       0:0
6    48       0:0
7    48       0:0
8    48       0:0
9    48       0:0
10   48       0:0
11   48       0:0
12   48       0:0
13   48       0:0
14   48       0:0
15   48       0:0
16   48       0:0
17   48       0:0
18   48       0:0
19   48       0:0
       day timestamp
77279   49       2:0
77280   49       2:0
77281   49       2:0
77282   49       2:0
77283   49       2:0
77284   49       2:0
77285   49       2:0
77286   49       2:0
77287   49       2:0
77288   49       2:0
77289   49       2:0
77290   49       2:0
77291   49       2:0
77292   49       2:0
77293   49       2:0
77294   49       2:0
77295   49       2:0
77296   49       2:0
77297   49       2:0
77298   49       2:0
